# Topo-Brain Validation Pipeline

**Checkpoint: 110,000 iterations**  
**Test subject: sub-06 (only held-out subject)**

Run each section in order.

## Steps
1. Setup & Paths
2. Step 1: Verify Test Split
3. Step 2: Full-Volume Evaluation (SSIM/PSNR/Dice/HD95)
4. Step 3: Aggregate Metrics
5. Step 4: MRIQC
6. Step 5: FastSurfer — Hippocampal Segmentation
7. Step 6: Blinded Reader Study Prep
8. Step 7: Regional Temporal Lobe Analysis
9. Final Results Summary

---
## 0. Install packages (run once)

In [ ]:
import subprocess, sys

packages = ['nibabel', 'scikit-image', 'tqdm', 'pyyaml', 'scipy', 'matplotlib']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('Packages ready.')

---
## 1. Setup & Paths

**Edit ONLY these paths, then run this cell.**

In [ ]:
import os
import json
import subprocess
import sys
from pathlib import Path

# ── EDIT THESE ────────────────────────────────────────────────
PROJECT_ROOT         = Path('/eos/user/p/ppokhrel/Untitled Folder 1/Topo-Brain')
DATA_ROOT            = Path('/eos/user/p/ppokhrel/Untitled Folder 1/preprocessed-mri-aligned-diffusion')
FS_LICENSE           = Path('/eos/user/p/ppokhrel/license.txt')
APPTAINER_MRIQC      = Path('/eos/user/p/ppokhrel/Untitled Folder 1/images/mriqc.sif')
APPTAINER_FASTSURFER = Path('/eos/user/p/ppokhrel/Untitled Folder 1/images/fastsurfer.sif')
# ──────────────────────────────────────────────────────────────

PAIRS_CSV     = PROJECT_ROOT / 'pairs_new.csv'
RESULTS_DIR   = PROJECT_ROOT / 'results'
CHECKPOINT_PT = PROJECT_ROOT / 'output' / 'checkpoint_110000' / 'checkpoint_110000.pt'
TEST_SUBJECTS = ['sub-06']

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

checks = {
    'Project root':   PROJECT_ROOT,
    'Data root':      DATA_ROOT,
    'Pairs CSV':      PAIRS_CSV,
    'Checkpoint':     CHECKPOINT_PT,
    'MRIQC image':    APPTAINER_MRIQC,
    'FastSurfer img': APPTAINER_FASTSURFER,
}
for label, path in checks.items():
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {label}: {path}')

print(f'\nTest subject: {TEST_SUBJECTS}')


---
## Step 1: Verify Test Split

In [ ]:
with open(PROJECT_ROOT / 'test_split.json') as f:
    split = json.load(f)

print(f"Train: {split['train']}")
print(f"Val/Test: {split['test']}")
print(f"Note: {split['_note']}")

---
## Step 2: Full-Volume Evaluation (SSIM / PSNR / Dice / HD95)

**Time: ~15-60 min depending on GPU availability.**  
If this is too slow in SWAN, run it from the lxplus terminal instead (see instructions below the cell).

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU — inference will be slow but will complete.')

In [ ]:
# Check if already done
metrics_file = RESULTS_DIR / 'sub-06' / 'metrics.json'
if metrics_file.exists():
    with open(metrics_file) as f:
        m = json.load(f)
    print('Evaluation already complete:')
    print(f"  SSIM: {m.get('ssim', 'N/A'):.4f}")
    print(f"  PSNR: {m.get('psnr', 'N/A'):.2f} dB")
    print(f"  Dice: {m.get('dice', 'N/A'):.4f}")
    print(f"  HD95: {m.get('hd95_mm', 'N/A'):.2f} mm")
    print('\nSkip to Step 3.')
else:
    print('Not yet evaluated. Running now...')
    cmd = [
        sys.executable, 'scripts/evaluate_full_volume.py',
        '--checkpoint', str(CHECKPOINT_PT),
        '--subject', 'sub-06',
        '--pairs_csv', str(PAIRS_CSV),
        '--data-root', str(DATA_ROOT),
        '--output_dir', str(RESULTS_DIR / 'sub-06'),
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT)
    if result.returncode == 0:
        print('Done.')
    else:
        print('ERROR — check output above.')

**Alternative — run from lxplus terminal if SWAN is too slow:**
```bash
cd /eos/home-i04/p/ppokhrel/topobrain/Topo-Brain
python scripts/evaluate_full_volume.py \
    --checkpoint output/checkpoint_110000/checkpoint_110000.pt \
    --subject sub-06 \
    --pairs_csv pairs_new.csv \
    --data-root /eos/home-i04/p/ppokhrel/data/topobrain/ \
    --output_dir results/sub-06/
```

---
## Step 3: Aggregate Metrics

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/aggregate_metrics.py',
     '--results-dir', str(RESULTS_DIR),
     '--subjects', 'sub-06',
     '--output', str(RESULTS_DIR / 'test_metrics.csv')],
    cwd=PROJECT_ROOT
)

csv_path = RESULTS_DIR / 'test_metrics.csv'
if csv_path.exists():
    print(csv_path.read_text())

---
## Step 4: MRIQC — Image Quality Metrics

Compares CJV, CNR, EFC, FBER, SNR across 3T / synthetic 7T / real 7T.  
Uses Apptainer (CERN does not have Docker).

In [ ]:
# Check Apptainer image exists
if APPTAINER_MRIQC.exists():
    print(f'MRIQC image found: {APPTAINER_MRIQC}')
else:
    print(f'MRIQC image NOT found: {APPTAINER_MRIQC}')
    print('\nPull it first — run this in the SWAN Terminal:')
    print(f'  mkdir -p {APPTAINER_MRIQC.parent}')
    print(f'  apptainer pull {APPTAINER_MRIQC} docker://nipreps/mriqc:latest')

In [ ]:
# Run MRIQC on 3T
subprocess.run(
    [sys.executable, 'scripts/run_mriqc.py',
     '--mode', '3t', '--subjects', 'sub-06',
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / '3t'),
     '--singularity', '--singularity-image', str(APPTAINER_MRIQC)],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run MRIQC on synthetic 7T
subprocess.run(
    [sys.executable, 'scripts/run_mriqc.py',
     '--mode', 'synthetic7t', '--subjects', 'sub-06',
     '--results-dir', str(RESULTS_DIR),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / 'synthetic7t'),
     '--singularity', '--singularity-image', str(APPTAINER_MRIQC)],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run MRIQC on real 7T
subprocess.run(
    [sys.executable, 'scripts/run_mriqc.py',
     '--mode', 'real7t', '--subjects', 'sub-06',
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--output-dir', str(RESULTS_DIR / 'mriqc' / 'real7t'),
     '--singularity', '--singularity-image', str(APPTAINER_MRIQC)],
    cwd=PROJECT_ROOT
)

In [ ]:
# Compare all three — prints the IQM table
subprocess.run(
    [sys.executable, 'scripts/run_mriqc.py',
     '--mode', 'compare',
     '--mriqc-dir', str(RESULTS_DIR / 'mriqc')],
    cwd=PROJECT_ROOT
)

---
## Step 5: FastSurfer — Hippocampal / MTL Segmentation

Needs: FreeSurfer license at `FS_LICENSE` path set in Setup cell.

In [ ]:
# Check Apptainer image and license
for label, path in [('FastSurfer image', APPTAINER_FASTSURFER), ('FS license', FS_LICENSE)]:
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {label}: {path}')

if not APPTAINER_FASTSURFER.exists():
    print('\nPull FastSurfer — run in SWAN Terminal:')
    print(f'  apptainer pull {APPTAINER_FASTSURFER} docker://deepmi/fastsurfer:latest')

if not FS_LICENSE.exists():
    print('\nGet FreeSurfer license (free):')
    print('  https://surfer.nmr.mgh.harvard.edu/registration.html')
    print(f'  Upload via CERNBox to: {FS_LICENSE}')

In [ ]:
FASTSURFER_DIR = RESULTS_DIR / 'fastsurfer'

# Run on synthetic 7T
subprocess.run(
    [sys.executable, 'scripts/run_fastsurfer.py',
     '--mode', 'synthetic7t', '--subjects', 'sub-06',
     '--results-dir', str(RESULTS_DIR),
     '--fastsurfer-dir', str(FASTSURFER_DIR),
     '--fs-license', str(FS_LICENSE),
     '--singularity', '--singularity-image', str(APPTAINER_FASTSURFER),
     '--seg-only'],
    cwd=PROJECT_ROOT
)

In [ ]:
# Run on real 7T
subprocess.run(
    [sys.executable, 'scripts/run_fastsurfer.py',
     '--mode', 'real7t', '--subjects', 'sub-06',
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--fastsurfer-dir', str(FASTSURFER_DIR),
     '--fs-license', str(FS_LICENSE),
     '--singularity', '--singularity-image', str(APPTAINER_FASTSURFER),
     '--seg-only'],
    cwd=PROJECT_ROOT
)

In [ ]:
# Compare hippocampal volumes: synthetic vs real 7T
subprocess.run(
    [sys.executable, 'scripts/run_fastsurfer.py',
     '--mode', 'compare', '--subjects', 'sub-06',
     '--fastsurfer-dir', str(FASTSURFER_DIR)],
    cwd=PROJECT_ROOT
)

---
## Step 6: Blinded Reader Study Prep

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/blinded_reader_prep.py',
     '--subjects', 'sub-06',
     '--data-root', str(DATA_ROOT),
     '--pairs-csv', str(PAIRS_CSV),
     '--results-dir', str(RESULTS_DIR),
     '--output-dir', str(RESULTS_DIR / 'blinded_study')],
    cwd=PROJECT_ROOT
)
print('Panels saved to results/blinded_study/panels/')
print('Download via CERNBox to view them.')

---
## Step 7: Regional Temporal Lobe Analysis

In [ ]:
import numpy as np

morph_file = RESULTS_DIR / 'fastsurfer' / 'morphometry_comparison.json'

if not morph_file.exists():
    print('Run Step 5 (FastSurfer) first.')
else:
    with open(morph_file) as f:
        morph = json.load(f)

    temporal_regions = [
        'lh_entorhinal_vol_mm3', 'rh_entorhinal_vol_mm3',
        'lh_parahippocampal_vol_mm3', 'rh_parahippocampal_vol_mm3',
        'lh_entorhinal_thick_mm', 'rh_entorhinal_thick_mm',
    ]

    print(f'{"Region":<40} {"Synthetic":>12} {"Real 7T":>12} {"Diff %":>10}')
    print('-' * 76)

    for metric in temporal_regions:
        s = morph.get('synthetic7t', {}).get('sub-06', {}).get(metric)
        r = morph.get('real7t', {}).get('sub-06', {}).get(metric)
        if s is None or r is None:
            continue
        diff = 100 * (s - r) / r if r else float('nan')
        flag = ' !' if abs(diff) > 10 else ''
        print(f'{metric:<40} {s:>12.1f} {r:>12.1f} {diff:>+10.1f}%{flag}')

---
## Final Results Summary

In [ ]:
print('=' * 65)
print('TOPO-BRAIN VALIDATION — CHECKPOINT 110k — sub-06')
print('=' * 65)

# 1. Quantitative metrics
print('\n1. SSIM / PSNR / Dice / HD95')
mf = RESULTS_DIR / 'sub-06' / 'metrics.json'
if mf.exists():
    m = json.loads(mf.read_text())
    print(f"   SSIM : {m.get('ssim', 'N/A'):.4f}   (target >0.90)")
    print(f"   PSNR : {m.get('psnr', 'N/A'):.2f} dB  (target >33 dB)")
    print(f"   Dice : {m.get('dice', 'N/A'):.4f}   (target >0.85)")
    print(f"   HD95 : {m.get('hd95_mm', 'N/A'):.2f} mm   (target <5 mm)")
else:
    print('   Not yet computed — run Step 2.')

# 2. MRIQC
print('\n2. MRIQC IQMs')
iqm_csv = RESULTS_DIR / 'mriqc' / 'iqm_comparison.csv'
if iqm_csv.exists():
    print(iqm_csv.read_text()[:800])
else:
    print('   Not yet computed — run Step 4.')

# 3. Hippocampal volumes
print('\n3. HIPPOCAMPAL VOLUMES')
mf2 = RESULTS_DIR / 'fastsurfer' / 'morphometry_comparison.json'
if mf2.exists():
    morph = json.loads(mf2.read_text())
    s = morph.get('synthetic7t', {}).get('sub-06', {})
    r = morph.get('real7t', {}).get('sub-06', {})
    print(f"   Hippocampus total — Synthetic: {s.get('hippocampus_total_mm3', 'N/A')}  Real: {r.get('hippocampus_total_mm3', 'N/A')} mm3")
else:
    print('   Not yet computed — run Step 5.')

# 4. Blinded study
print('\n4. BLINDED READER STUDY')
bs = RESULTS_DIR / 'blinded_study' / 'scoresheet.csv'
print('   Panels ready.' if bs.exists() else '   Not yet prepared — run Step 6.')

print()
print('=' * 65)